<a id="intro"></a>
# ЛР № 5. Расчёт метрик почасового спроса Seoul Bike

Это первая часть общего аналитического ноутбука ЛР № 5 и № 7.
Здесь читаем готовую витрину `mart`, рассчитываем метрики и проверяем результаты.
В седьмой работе ниже добавляются визуализации. Промежуточные файлы данных не создаются.

Все обязательные описания, определения метрик и вывод собраны в
[docs/metrics.md](../docs/metrics.md). В ноутбуке — работающий код, SQL,
результаты расчётов и проверок.

Предыдущий этап: [витрина ЛР № 4](../docs/mart.md).
Сквозной вопрос: [Project Brief](../docs/project_brief.md).

## 1. Вход и параметры

Выбранный вопрос из `Project Brief`: **как различается почасовой спрос на велосипеды
по сезону, часу суток и типу дня?**

Период — весь снимок: с 01.12.2017 по 30.11.2018, поэтому отдельный фильтр по периоду
не задаётся. Логическая дата в пути витрины — это дата загрузки исходных данных,
а не время наблюдения.

Ноутбук запускается из корня репозитория или каталога `notebooks` с ядром проектного
`.venv`. Стенд должен работать, настройки S3 читаются из `.env`. Повторный запуск
первой части полностью восстанавливает результаты после перезапуска ядра.

In [1]:
# M01. Параметры
DATASET_SLUG = "seoul_bike"
SOURCE_INGESTED_ON = "2026-11-12"
MART_PATH = (
    f"s3://mart/{DATASET_SLUG}/source_ingested_on={SOURCE_INGESTED_ON}/hourly_demand.parquet"
)
print(f"Вход: {MART_PATH}")

Вход: s3://mart/seoul_bike/source_ingested_on=2026-11-12/hourly_demand.parquet


## 2. Подключение

Переиспользуем подключение из предыдущих работ. Настройки читаются из `.env`.
Вспомогательная функция только выполняет SQL и выводит таблицу; вся логика расчётов
расположена в явных запросах ниже.

In [2]:
# M02. Подключение
import os
import sys
from html import escape
from pathlib import Path

import duckdb
from dotenv import load_dotenv
from IPython.display import HTML, display

PROJECT_ROOT = next(
    parent for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / "pyproject.toml").is_file()
    and (parent / "src/analysis/read_raw.py").is_file()
)
load_dotenv(PROJECT_ROOT / ".env", override=True)
settings = ("S3_ENDPOINT", "S3_ACCESS_KEY", "S3_SECRET_KEY")
missing = [name for name in settings if not os.getenv(name)]
if missing:
    raise RuntimeError(f"Не заданы настройки: {', '.join(missing)}")
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from analysis.read_raw import configure_s3  # noqa: E402

connection = duckdb.connect()
configure_s3(connection)


def literal(value):
    return "'" + str(value).replace("'", "''") + "'"


def select_table(sql):
    cursor = connection.execute(sql)
    description = cursor.description
    if description is None:
        raise ValueError("Ожидается таблица результата")
    rows = cursor.fetchall()
    header = "".join(f"<th>{escape(column[0])}</th>" for column in description)
    body = "".join(
        "<tr>" + "".join(f"<td>{escape(str(item))}</td>" for item in row) + "</tr>"
        for row in rows
    )
    display(HTML(f"<table><thead><tr>{header}</tr></thead><tbody>{body}</tbody></table>"))
    print(f"Строк результата: {len(rows)}")
    return rows

## 3. Чтение витрины

Одна строка `hourly_demand` — один час работы системы велопроката в определённую дату.
Надёжного бизнес-ключа в источнике нет; это решение ЛР № 3–4.

`analysis_base` один раз загружает витрину из S3 в память DuckDB. Следующие расчёты
и будущие графики используют данные в памяти этого подключения.

**Решение о качестве, важное для метрик.** В ЛР № 3 установлено, что нулевой спрос
означает остановку сервиса: все 295 таких часов имеют `functioning_day = false`.
В ЛР № 4 они сохранены в витрине с признаком `service_closed`. Поэтому метрики спроса
считаются **по открытым часам** (`is_open`), а число всех часов показывается рядом:
без этого разделения средний спрос оказался бы занижен.

In [3]:
# M03. Вход
connection.execute(f"""
    CREATE OR REPLACE TEMP VIEW demand AS
    SELECT * FROM read_parquet({literal(MART_PATH)}, hive_partitioning = false)
""")
required_fields = {
    "observed_on", "hour", "rented_bike_count", "season", "day_type",
    "is_open", "service_closed", "temperature_c", "warm_snowfall_flag",
}
schema = {row[0]: row[1] for row in connection.execute("DESCRIBE demand").fetchall()}
assert not (required_fields - schema.keys()), "В витрине отсутствуют нужные поля"

connection.execute("""
    CREATE OR REPLACE TEMP TABLE analysis_base AS
    SELECT * FROM demand
""")
input_rows = select_table("""
    SELECT
        count(*) AS hours_count,
        count(*) FILTER (WHERE is_open) AS open_hours,
        count(*) FILTER (WHERE service_closed) AS closed_hours,
        count(DISTINCT observed_on) AS days_count,
        min(observed_on) AS first_day,
        max(observed_on) AS last_day
    FROM analysis_base
""")
print(f"Физических полей витрины: {len(schema)}")

hours_count,open_hours,closed_hours,days_count,first_day,last_day
8760,8465,295,365,2017-12-01,2018-11-30


Строк результата: 1
Физических полей витрины: 19


<a id="definitions"></a>
## 4. Что считаем

**Вопрос:** как различается почасовой спрос на велосипеды по сезону, часу суток
и типу дня?

Три разные метрики:

1. **Число открытых часов** — `count(*) FILTER (WHERE is_open)`. Сколько наблюдений
   вошло в группу. Это размер группы и одновременно знаменатель для среднего.
2. **Средний спрос за час** — `avg(rented_bike_count) FILTER (WHERE is_open)`,
   в арендах за час. Считается только по открытым часам.
3. **Медианный спрос за час** — `median(rented_bike_count) FILTER (WHERE is_open)`,
   в арендах за час. Устойчив к выбросам, в отличие от среднего.

**Почему это три разные метрики, а не одна в разных группах.** Число часов измеряет
объём наблюдений, среднее — типичную нагрузку, медиана — типичную нагрузку при
скошенном распределении. Они отвечают на разные части вопроса: насколько велика группа,
какова в ней средняя нагрузка и какова нагрузка в «обычный» час.

**Оконная функция.** `percent_of_hours` — доля часов группы в общем числе часов периода,
в процентах: `100.0 * open_hours / NULLIF(SUM(open_hours) OVER (), 0)`. Знаменатель
защищён `NULLIF` от нуля. Значение показывает, насколько равномерно часы распределены
по срезам: если доля группы мала, её метрики опираются на меньшее число наблюдений
и сравнение менее устойчиво.

Рядом с каждой метрикой выводится `open_hours` — число значений, вошедших в среднее
и медиану. Отдельно сохраняется `rented_sum` (сумма аренд группы) — она нужна для
проверки того, что общее среднее нельзя получить усреднением средних групп.

## 5. Общий итог

Фильтр по `is_open` находится у конкретных агрегатов, а не в общем `WHERE`: иначе
из расчёта выпали бы часы остановки сервиса и общее число наблюдений перестало бы
сходиться с витриной.

In [4]:
# M04. Общие метрики
connection.execute("""
    CREATE OR REPLACE TEMP TABLE overall_metrics AS
    SELECT
        count(*) AS hours_count,
        count(*) FILTER (WHERE is_open) AS open_hours,
        count(*) FILTER (WHERE service_closed) AS closed_hours,
        sum(rented_bike_count) AS rented_sum,
        avg(rented_bike_count) FILTER (WHERE is_open) AS avg_per_hour,
        median(rented_bike_count) FILTER (WHERE is_open) AS median_per_hour,
        avg(rented_bike_count) AS avg_including_closed,
        median(rented_bike_count) AS median_including_closed
    FROM analysis_base
""")
overall_rows = select_table("""
    SELECT hours_count, open_hours, closed_hours, rented_sum,
           round(avg_per_hour, 2) AS avg_per_hour,
           round(median_per_hour, 1) AS median_per_hour,
           round(avg_including_closed, 2) AS avg_including_closed,
           round(median_including_closed, 1) AS median_including_closed
    FROM overall_metrics
""")

overall_raw = connection.execute("SELECT * FROM overall_metrics").fetchone()
assert overall_raw is not None

hours_count,open_hours,closed_hours,rented_sum,avg_per_hour,median_per_hour,avg_including_closed,median_including_closed
8760,8465,295,6172314,729.16,542.0,704.6,504.5


Строк результата: 1


## 6. Основная группировка: по часу суток, с оконной функцией

Одна строка результата — час суток (0–23) за все дни периода. Оконная функция добавляет
долю часов группы в общем числе часов, сохраняя все 24 строки.

In [5]:
# M05. Основная группировка по часам
connection.execute("""
    CREATE OR REPLACE TEMP TABLE hourly_metrics AS
    WITH grouped AS (
        SELECT
            hour,
            count(*) AS hours_count,
            count(*) FILTER (WHERE is_open) AS open_hours,
            count(*) FILTER (WHERE service_closed) AS closed_hours,
            sum(rented_bike_count) AS rented_sum,
            avg(rented_bike_count) FILTER (WHERE is_open) AS avg_per_hour,
            median(rented_bike_count) FILTER (WHERE is_open) AS median_per_hour
        FROM analysis_base
        GROUP BY hour
    )
    SELECT
        *,
        100.0 * open_hours / nullif(sum(open_hours) OVER (), 0) AS percent_of_hours
    FROM grouped
""")
hourly_rows = select_table("""
    SELECT hour, hours_count, open_hours, closed_hours,
           round(avg_per_hour, 1) AS avg_per_hour,
           round(median_per_hour, 1) AS median_per_hour,
           round(percent_of_hours, 4) AS percent_of_hours
    FROM hourly_metrics
    ORDER BY hour
""")
hourly_raw = connection.execute("""
    SELECT * FROM hourly_metrics ORDER BY hour
""").fetchall()

hour,hours_count,open_hours,closed_hours,avg_per_hour,median_per_hour,percent_of_hours
0,365,352,13,561.5,531.5,4.1583
1,365,352,13,441.9,411.5,4.1583
2,365,352,13,312.8,277.0,4.1583
3,365,352,13,210.8,183.5,4.1583
4,365,352,13,137.5,123.5,4.1583
5,365,352,13,144.2,136.5,4.1583
6,365,352,13,298.2,245.0,4.1583
7,365,353,12,626.6,443.0,4.1701
8,365,353,12,1050.2,767.0,4.1701
9,365,353,12,667.9,719.0,4.1701


Строк результата: 24


## 7. Уточняющая группировка: тип дня и час

Добавляем признак `day_type` к группировке, сохраняя те же определения метрик. Таблица
помогает ответить на тот же вопрос: показывает, различается ли форма суточного профиля
между рабочим, выходным и праздничным днём.

Временная таблица содержит все имеющиеся сочетания типа дня и часа — их 72 (3 × 24).
Отсутствующие сочетания не подменяются нулём.

In [6]:
# M06. Уточняющая группировка
connection.execute("""
    CREATE OR REPLACE TEMP TABLE segment_metrics AS
    WITH grouped AS (
        SELECT
            day_type,
            hour,
            count(*) AS hours_count,
            count(*) FILTER (WHERE is_open) AS open_hours,
            count(*) FILTER (WHERE service_closed) AS closed_hours,
            sum(rented_bike_count) AS rented_sum,
            avg(rented_bike_count) FILTER (WHERE is_open) AS avg_per_hour,
            median(rented_bike_count) FILTER (WHERE is_open) AS median_per_hour
        FROM analysis_base
        GROUP BY day_type, hour
    )
    SELECT
        *,
        100.0 * open_hours / nullif(sum(open_hours) OVER (PARTITION BY day_type), 0)
            AS percent_within_day_type
    FROM grouped
""")
print("Первые 12 строк группировки по типу дня и часу:")
segment_preview = select_table("""
    SELECT day_type, hour, hours_count, open_hours,
           round(avg_per_hour, 1) AS avg_per_hour,
           round(median_per_hour, 1) AS median_per_hour
    FROM segment_metrics
    ORDER BY day_type, hour
    LIMIT 12
""")
segment_totals = select_table("""
    SELECT day_type,
           sum(hours_count) AS hours_count,
           sum(open_hours) AS open_hours
    FROM segment_metrics
    GROUP BY day_type
    ORDER BY open_hours DESC, day_type
""")
segment_rows = connection.execute("""
    SELECT * FROM segment_metrics ORDER BY day_type, hour
""").fetchall()
print(f"Строк уточняющей группировки: {len(segment_rows)}")

Первые 12 строк группировки по типу дня и часу:


day_type,hour,hours_count,open_hours,avg_per_hour,median_per_hour
выходной,0,100,97,685.1,720.0
выходной,1,100,97,560.0,578.0
выходной,2,100,97,419.2,428.0
выходной,3,100,97,289.4,299.0
выходной,4,100,97,178.2,185.0
выходной,5,100,97,128.7,112.0
выходной,6,100,97,168.2,150.0
выходной,7,100,98,262.1,261.5
выходной,8,100,98,431.8,460.0
выходной,9,100,98,522.6,571.0


Строк результата: 12


day_type,hours_count,open_hours
рабочий,5928,5712
выходной,2400,2345
праздник,432,408


Строк результата: 3
Строк уточняющей группировки: 72


## 8. Отдельные числовые значения для дальнейшего анализа

Эти выборки нужны для распределения и исследования связи. Восстановить их из средних
или медиан групп нельзя, поэтому значения берутся построчно.

- `distribution_values` — спрос каждого открытого часа: для распределения.
- `relationship_values` — пара «температура, спрос» по открытым часам: для исследования
  связи. Погодные значения с флагами ЛР № 3 исключаются: 17 часов с нулевой влажностью
  и 14 со снегом при тёплой температуре признаны сомнительными. Здесь исключается
  только пара «температура — спрос», сами часы в других метриках остаются.

In [7]:
# M07. Отдельные значения
connection.execute("""
    CREATE OR REPLACE TEMP TABLE distribution_values AS
    SELECT observed_on, hour, day_type, rented_bike_count
    FROM analysis_base
    WHERE is_open
""")
connection.execute("""
    CREATE OR REPLACE TEMP TABLE relationship_values AS
    SELECT observed_on, hour, day_type, temperature_c, rented_bike_count
    FROM analysis_base
    WHERE is_open AND NOT warm_snowfall_flag
""")
distribution_summary = select_table("""
    SELECT
        count(*) AS open_hours,
        min(rented_bike_count) AS min_demand,
        quantile_cont(rented_bike_count, 0.25) AS p25_demand,
        median(rented_bike_count) AS median_demand,
        quantile_cont(rented_bike_count, 0.75) AS p75_demand,
        max(rented_bike_count) AS max_demand
    FROM distribution_values
""")
relationship_summary = select_table("""
    SELECT
        count(*) AS pairs_count,
        round(corr(rented_bike_count, temperature_c), 4) AS pearson_r,
        min(temperature_c) AS min_temperature,
        max(temperature_c) AS max_temperature
    FROM relationship_values
""")
durations = [
    row[0] for row in connection.execute("""
        SELECT rented_bike_count FROM distribution_values ORDER BY rented_bike_count
    """).fetchall()
]
demand_values = durations
temperature_pairs = connection.execute("""
    SELECT temperature_c, rented_bike_count FROM relationship_values
""").fetchall()

open_hours,min_demand,p25_demand,median_demand,p75_demand,max_demand
8465,2,214.0,542.0,1084.0,3556


Строк результата: 1


pairs_count,pearson_r,min_temperature,max_temperature
8451,0.5627,-17.8,39.4


Строк результата: 1


<a id="validation"></a>
## 9. Проверки расчётов

Пять проверок. Для денежных и дробных сравнений используется небольшой допуск.
Проверка 5 подтверждает, что общее среднее нельзя получить простым усреднением
средних по группам: она объединяет средние с весами `open_hours`.

In [8]:
# M08. Проверки
check_rows = select_table("""
    WITH hourly AS (
        SELECT hour, open_hours, rented_sum, avg_per_hour,
               percent_of_hours
        FROM hourly_metrics
    ), segment AS (
        SELECT day_type, hour, open_hours, rented_sum
        FROM segment_metrics
    ), overall AS (
        SELECT open_hours, hours_count, rented_sum, avg_per_hour
        FROM overall_metrics
    ), checks AS (
        SELECT 1 AS check_no,
               'Сумма открытых часов по часам равна общему итогу' AS check_name,
               (SELECT sum(open_hours) FROM hourly)::DOUBLE AS actual,
               (SELECT open_hours FROM overall)::DOUBLE AS expected
        UNION ALL
        SELECT 2,
               'Сумма открытых часов по типам дня и часам равна общему итогу',
               (SELECT sum(open_hours) FROM segment)::DOUBLE,
               (SELECT open_hours FROM overall)::DOUBLE
        UNION ALL
        SELECT 3,
               'Сумма долей часов равна 100 процентам',
               (SELECT sum(percent_of_hours) FROM hourly)::DOUBLE,
               100.0
        UNION ALL
        SELECT 4,
               'Сумма аренд по часам равна общей сумме аренд',
               (SELECT sum(rented_sum) FROM hourly)::DOUBLE,
               (SELECT rented_sum FROM overall)::DOUBLE
        UNION ALL
        SELECT 5,
               'Среднее по записям совпадает со средним по часовым итогам с весами',
               (SELECT sum(avg_per_hour * open_hours) / nullif(sum(open_hours), 0)
                FROM hourly),
               (SELECT avg_per_hour FROM overall)
    )
    SELECT check_name,
           round(actual, 4) AS actual,
           round(expected, 4) AS expected,
           abs(actual - expected) < 1e-6 AS passed
    FROM checks
    ORDER BY check_no
""")
assert all(row[3] for row in check_rows), "Проверки не сошлись: вернитесь к расчётам"
print(f"Проверок пройдено: {len(check_rows)} из {len(check_rows)}")

check_name,actual,expected,passed
Сумма открытых часов по часам равна общему итогу,8465.0,8465.0,True
Сумма открытых часов по типам дня и часам равна общему итогу,8465.0,8465.0,True
Сумма долей часов равна 100 процентам,100.0,100.0,True
Сумма аренд по часам равна общей сумме аренд,6172314.0,6172314.0,True
Среднее по записям совпадает со средним по часовым итогам с весами,729.157,729.157,True


Строк результата: 5
Проверок пройдено: 5 из 5


<a id="continuation"></a>
## Продолжение в ЛР № 7

Первая часть завершена. Пояснения и численный вывод находятся в
[docs/metrics.md](../docs/metrics.md).

Подготовлены временные таблицы:

- `hourly_metrics` — основная группировка по часам суток с оконной функцией;
- `segment_metrics` — уточняющая группировка по типу дня и часу;
- `distribution_values` — спрос каждого открытого часа для распределения;
- `relationship_values` — пары «температура, спрос» для исследования связи;
- `overall_metrics` — общий итог.

Для работы из Python доступны `overall_raw`, `hourly_raw`, `segment_rows`,
`segment_totals`, `demand_values`, `temperature_pairs` и `distribution_summary` —
эти данные не округлены.

В седьмой работе ниже будут добавлены ячейки визуализации. После перезапуска ядра
сначала выполните M01–M08: все результаты должны восстанавливаться из кода,
а не зависеть от сохранившейся памяти.